# Memora AI - Code Error Analysis & Model Training Pipeline

This notebook connects directly to the **Memora AI Local Database** (`memoradb.db`), analyzes verified code defect submissions, and trains a supervised machine learning model to classify runtime and logic error types.

In [ ]:
# 1. Environment & Module Setup
import os
import sys
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Ensure project root is in python path
sys.path.insert(0, str(Path.cwd()))

from app.db.session import SessionLocal, engine, Base
from app.db.models import User, ErrorClassificationExample, CodeSubmission
from app.training.train_model import MODEL_PATH, train_error_classifier
from app.training.predict import predict_error_type

print("✓ Imports successful. Database engine ready.")

## 2. Load Dataset from Local SQLite Database (`memoradb.db`)

In [ ]:
# Connect to the database session
db = SessionLocal()

# Query all verified error classification examples
records = db.query(ErrorClassificationExample).filter(ErrorClassificationExample.is_verified.is_(True)).all()

data = []
for r in records:
    combined_text = f"{r.user_code}\n{r.stack_trace or ''}\n{r.pattern_cluster or ''}".strip()
    data.append({
        "id": r.id,
        "language": r.programming_language,
        "code": r.user_code,
        "stack_trace": r.stack_trace,
        "pattern_cluster": r.pattern_cluster,
        "error_type": r.error_type,
        "text": combined_text
    })

df = pd.DataFrame(data)
print(f"Loaded {len(df)} verified training samples from local database.")
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("=== Error Type Distribution ===")
print(df["error_type"].value_counts())

print("\n=== Programming Language Breakdown ===")
print(df["language"].value_counts())

## 4. Feature Extraction & Model Training Pipeline

We use a **TF-IDF Vectorizer** (unigrams + bigrams) paired with a multinomial **Logistic Regression** classifier.

In [ ]:
X = df["text"]
y = df["error_type"]

# Define end-to-end Machine Learning Pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ("clf", LogisticRegression(max_iter=1000, solver="lbfgs"))
])

# Fit model on local dataset
pipeline.fit(X, y)
train_preds = pipeline.predict(X)
acc = accuracy_score(y, train_preds)

print(f"✓ Model fitted on {len(df)} samples across {len(pipeline.classes_)} classes.")
print(f"✓ Training Accuracy: {acc * 100:.2f}%")
print("\n=== Classes Learned ===")
print(pipeline.classes_)

## 5. Model Evaluation & Classification Report

In [ ]:
print("=== Detailed Classification Report ===")
print(classification_report(y, train_preds, zero_division=0))

## 6. Save Trained Model to Backend Artifacts

Export the serialized model bundle so that the live FastAPI backend uses the updated weights.

In [ ]:
target_artifact_path = Path(MODEL_PATH)
target_artifact_path.parent.mkdir(parents=True, exist_ok=True)

with open(target_artifact_path, "wb") as f:
    pickle.dump(pipeline, f)

print(f"✓ Model artifact successfully serialized to: {target_artifact_path}")

## 7. Live Inference & Validation Checks

In [ ]:
test_samples = [
    ("int* ptr = NULL; *ptr = 100;", "C Null Pointer"),
    ("arr = [1, 2, 3]; print(arr[50])", "Python Index Out of Range"),
    ("f = open('output.txt'); f.write('log')", "Python Unclosed File Leak"),
    ("def infinite_loop(): return infinite_loop()", "Python Infinite Recursion")
]

print("=== Real-time Model Predictions ===")
for code_snippet, label in test_samples:
    prediction = predict_error_type(code_snippet, db)
    print(f"[{label}]\nCode: {code_snippet}\n-> Predicted Defect: {prediction}\n")

db.close()
print("✓ Database connection closed. Pipeline execution complete.")